In [ ]:
import argparse
import math
import re

from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from math import pi
from pathlib import Path
from typing import Any, Generator, List, Optional

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import PyKDL
import rclpy
import rosbag2_py

# import ros2_numpy as rnp
import rosbag2_py._storage as storage
import scipy
import tf2_geometry_msgs
import tf2_ros

from utils.analysis_utils import (
    df_to_pose_stamped,
    do_transform_twist,
    filter_dataframe,
    filter_twist_data,
    transform_pose_dataframe,
    transform_to_kdl,
    transform_twist_dataframe,
    quaternion_to_euler
)
from utils.bag_utils import get_metadata, get_reader
from builtin_interfaces.msg import Time
from control_msgs.msg import DynamicJointState
from geometry_msgs.msg import Pose, PoseStamped, TwistStamped, Vector3, Vector3Stamped
from mpl_toolkits.mplot3d import Axes3D
from numpy.typing import NDArray
from pandas import DataFrame, Series
from rclpy.serialization import deserialize_message
from utils.ros_to_df import ros_message_to_df
from rosidl_runtime_py.utilities import get_message
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.spatial.transform import Rotation as R
from sensor_msgs.msg import Imu
from std_msgs.msg import Header, String
from tf2_msgs.msg import TFMessage
from tf2_ros import TransformStamped
from utils.viz_utils import (
    MultiViconPoseTwistAnimator,
    publish_df_pose_stamped,
    publish_pose_stamped,
    publish_pose_stamped_list,
    send_static_transform,
    send_transform,
)

from motor_msgs.msg import MotorCurrents
import matplotlib.pyplot as plt
import multiprocessing
import time
import matplotlib
# matplotlib.use("Agg") 
# %matplotlib inline

from utils.deriv_smoothing import savgol_derivative, quaternion_derivative_savgol

plt.rcParams.update({
    "font.size": 16,  # Change all fonts to size 14
    "axes.titlesize": 16,  # Title font size
    "axes.labelsize": 14,  # Axis label font size
    "xtick.labelsize": 16,  # X-axis tick font size
    "ytick.labelsize": 16,  # Y-axis tick font size
    "legend.fontsize": 16   # Legend font size
})

ImportError: cannot import name 'solve_ivpConcrete' from 'scipy.integrate' (/home/rb/retail-bot/.venv/lib/python3.12/site-packages/scipy/integrate/__init__.py)

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


bag_files = [
    # "2025_2_16_rosbags/2025_2_16__16_00_08RPM5.0_Vinyl.bag"
    # '2025_3_11_rosbags_correct/2025_3_11__15_36_11RPM30_con.bag'
    # '2025_3_11_rosbags_correct/2025_3_11__15_49_15RPM20_con.bag'
    # 'Pickles/PICKLED_2025_3_11__15_34_08RPM60_con.bag_results.pkl'
    'Pickles/PICKLED_2025_3_11__15_25_25RPM60_Vinyl.bag_results.pkl'
    # '2025_3_11_rosbags_correct/2025_3_11__15_25_25RPM60_Vinyl.bag'
    # '2025_3_11_rosbags_correct/2025_3_11__15_27_48RPM20_Vinyl.bag'
]
bag_data = {}

for bag in bag_files:
    with open(f"{bag}", "rb") as f:
        p = pickle.load(f)
        bag_data[p['base_meta']['bag_name']] = p

In [ ]:
# results = bag_data['2025_3_11__15_34_08RPM60_con.bag']
results = bag_data['2025_3_11__15_25_25RPM60_Vinyl.bag']


In [ ]:
from scipy.stats import linregress
SURFACE_TYPE = "Vinyl"
%matplotlib qt
plt.close('all')

# Extract slip data
slip_x_adjusted = filter_dataframe(results["slip_x_adjusted"],
    columns_to_filter= ['bl_wheel_vel','fl_wheel_vel', 'br_wheel_vel', 'fr_wheel_vel'],
    cutoff=10, fs=30, order=4)

combined_currents_interp = results["combined_currents_interp"]

min_slip_threshold = 0.00
max_slip_threshold = 0.2

min_current_threshold = 0.05

# Keep only values within the valid slip range
slip_x_adjusted = slip_x_adjusted[
    (slip_x_adjusted.abs() >= min_slip_threshold)
    & (slip_x_adjusted.abs() <= max_slip_threshold)
]


# Ensure slip has no zeros to avoid division errors
safe_slip_x = slip_x_adjusted.replace(0, np.nan)

# slip_x_adjusted.plot(title=f"Subset Slip Over Time \n{SURFACE_TYPE}", ylabel="Slip Ratio")


fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
fig.suptitle(f"Motor Currents vs Slip with Best-Fit Lines \n{SURFACE_TYPE}")

# Correctly match motors with their respective wheels
motor_wheel_pairs = {
    "l_front": "fl_wheel_vel",
    "l_back": "bl_wheel_vel",
    "r_front": "fr_wheel_vel",
    "r_back": "br_wheel_vel",
}

# Find per-wheel slip-slope.
# for ax, (motor, wheel) in zip(axes.flatten(), motor_wheel_pairs.items()):
#     # Get valid non-NaN values for regression
#     # valid_mask = ~np.isnan(safe_slip_x[wheel]) & ~np.isnan(
#     #     combined_currents_interp[motor]
#     # )

#     valid_mask = (
#     ~np.isnan(safe_slip_x[wheel])
#     & ~np.isnan(combined_currents_interp[motor])
#     & (np.abs(combined_currents_interp[motor]) >= min_current_threshold)
#     )

#     slip_valid = safe_slip_x[wheel][valid_mask]
#     current_valid = combined_currents_interp[motor][valid_mask]

#     # Save per wheel data to results
#     results[f"{SURFACE_TYPE}_per_wheel_data"][f"{motor}"] = pd.DataFrame({
#         "slip": slip_valid.values,
#         "current": current_valid.values
#     })


#     if len(slip_valid) > 1:  # Ensure enough points for regression
#         # Compute linear regression (best-fit line)
#         slope, intercept, _, _, _ = linregress(slip_valid, current_valid)

#         # Generate regression line points
#         slip_fit = np.linspace(slip_valid.min(), slip_valid.max(), 100)
#         current_fit = slope * slip_fit + intercept

#         # Scatter plot
#         ax.scatter(slip_valid, current_valid, alpha=0.5, label=f"{motor} vs {wheel}")

#         # Best-fit line
#         ax.plot(
#             slip_fit, current_fit, linestyle="--", color="black", label="Best Fit Line"
#         )
#         # slope *= 4.23 # Make N-cm
#         # Add text box for slope value
#         textstr = f"Slope: {slope:.4f} Amps/Slip%"
#         ax.text(
#             0.65,
#             0.25,
#             textstr,
#             transform=ax.transAxes,
#             fontsize=10,
#             verticalalignment="top",
#             bbox=dict(facecolor="white", alpha=0.5),
#         )

#     ax.set_xlabel("Slip Ratio")
#     ax.set_ylabel("Current (A)")
#     ax.legend()
#     ax.grid(True)




# k = mu/slip
# mu = k * s
# 433.3g=cm/A
# So replace slope units with 
# 4.24 N-cm



# Aggregate all data.
combined_slip = []
combined_current = []

for motor, wheel in motor_wheel_pairs.items():
    # valid_mask = ~np.isnan(safe_slip_x[wheel]) & ~np.isnan(
    #     combined_currents_interp[motor]
    # )

    valid_mask = (
    ~np.isnan(safe_slip_x[wheel])
    & ~np.isnan(combined_currents_interp[motor])
    & (np.abs(combined_currents_interp[motor]) >= min_current_threshold)
    )

    combined_slip.extend(safe_slip_x[wheel][valid_mask])
    combined_current.extend(combined_currents_interp[motor][valid_mask])

# Only convert to NumPy arrays AFTER aggregation
combined_slip = np.array(combined_slip)
combined_current = np.array(combined_current)



# Compute single regression for all wheels.
if len(combined_slip) > 1:
    slope, intercept, _, _, _ = linregress(combined_slip, combined_current)
    slip_fit = np.linspace(combined_slip.min(), combined_slip.max(), 100)
    current_fit = slope * slip_fit + intercept

    # **Step 4: Plot combined results**
    plt.figure(figsize=(10, 6))
    plt.scatter(
        combined_slip, combined_current, alpha=0.5, label="All Data", color="blue"
    )
    plt.plot(
        slip_fit, current_fit, linestyle="--", color="black", label="Best Fit Line"
    )
    # slope *= 4.23 # Make N-cm
    # Add text box for slope value
    textstr = f"Slope: {slope:.4f} Amps/Slip%"
    plt.text(
        0.65,
        0.25,
        textstr,
        transform=plt.gca().transAxes,
        fontsize=16,
        verticalalignment="top",
        bbox=dict(facecolor="white", alpha=0.5),
    )

    plt.xlabel("Slip Ratio")
    plt.ylabel("Current (A)")
    plt.title(f"Average Motor Currents vs Slip with Best-Fit Line \n{SURFACE_TYPE}")
    # plt.legend()
    plt.grid(True)

wheel_titles = {
    "fl_wheel_vel": "Front Left Wheel",
    "bl_wheel_vel": "Back Left Wheel",
    "fr_wheel_vel": "Front Right Wheel",
    "br_wheel_vel": "Back Right Wheel",
}

QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-rb'
glx: failed to create dri3 screen
failed to load driver: nouveau
glx: failed to create dri3 screen
failed to load driver: nouveau


In [ ]:
# plt.close('all')


wheel_titles = {
    "fl_wheel_vel": "Front Left Wheel",
    "bl_wheel_vel": "Back Left Wheel",
    "fr_wheel_vel": "Front Right Wheel",
    "br_wheel_vel": "Back Right Wheel",
}

for ax, (motor, wheel) in zip(axes.flatten(), motor_wheel_pairs.items()):
    valid_mask = (
        ~np.isnan(safe_slip_x[wheel])
        & ~np.isnan(combined_currents_interp[motor])
        & (np.abs(combined_currents_interp[motor]) >= min_current_threshold)
    )

    slip_valid = safe_slip_x[wheel][valid_mask]
    current_valid = combined_currents_interp[motor][valid_mask]

    # Save per wheel data to results
    results[f"{SURFACE_TYPE}_per_wheel_data"][f"{motor}"] = pd.DataFrame({
        "slip": slip_valid.values,
        "current": current_valid.values
    })

    if len(slip_valid) > 1:
        slope, intercept, _, _, _ = linregress(slip_valid, current_valid)
        slip_fit = np.linspace(slip_valid.min(), slip_valid.max(), 100)
        current_fit = slope * slip_fit + intercept

        ax.scatter(slip_valid, current_valid, alpha=0.5)
        ax.plot(slip_fit, current_fit, linestyle="--", color="black")

        slope *= 4.23  # Convert to N-cm
        textstr = f"Slope: {slope:.4f} Amps/Slip%"
        ax.text(
            0.45,
            0.25,
            textstr,
            transform=ax.transAxes,
            fontsize=12,
            verticalalignment="top",
            bbox=dict(facecolor="white", alpha=0.5),
        )

    ax.set_title(wheel_titles[wheel])  # <- Title per subplot
    ax.set_xlabel("Slip Ratio")
    ax.set_ylabel("Current (A)")
    ax.grid(True)
    ax.legend().remove()  # <- Remove legend



No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.


In [ ]:

# # Conversion constants
# K_Ncm_per_A = 1300 * 9.81 * 1e-3  # g-cm/A -> N-cm/A
# wheel_radius_cm = 0.33 * 100 / 2  # m -> cm
# normal_force_N = 38.48  # N per wheel

# # μ = (Torque/r) / F_normal = (K / r) / F_normal
# conversion_factor = (K_Ncm_per_A / wheel_radius_cm) / normal_force_N

# for ax, (motor, wheel) in zip(axes.flatten(), motor_wheel_pairs.items()):
#     valid_mask = (
#         ~np.isnan(safe_slip_x[wheel])
#         & ~np.isnan(combined_currents_interp[motor])
#         & (np.abs(combined_currents_interp[motor]) >= min_current_threshold)
#     )

#     slip_valid = safe_slip_x[wheel][valid_mask]
#     current_valid = combined_currents_interp[motor][valid_mask]

#     # Convert currents to COF μ units
#     mu_valid = current_valid * conversion_factor

#     # Save per wheel data to results
#     results[f"{SURFACE_TYPE}_per_wheel_data"][f"{motor}"] = pd.DataFrame({
#         "slip": slip_valid.values,
#         "mu": mu_valid.values
#     })

#     if len(slip_valid) > 1:
#         slope, intercept, _, _, _ = linregress(slip_valid, mu_valid)
#         slip_fit = np.linspace(slip_valid.min(), slip_valid.max(), 100)
#         mu_fit = slope * slip_fit + intercept

#         ax.scatter(slip_valid, mu_valid, alpha=0.5)
#         ax.plot(slip_fit, mu_fit, linestyle="--", color="black")

#         textstr = f"Slope: {slope:.4f} μ/Slip%"
#         ax.text(
#             0.65,
#             0.25,
#             textstr,
#             transform=ax.transAxes,
#             fontsize=10,
#             verticalalignment="top",
#             bbox=dict(facecolor="white", alpha=0.5),
#         )

#     ax.set_title(wheel_titles[wheel])
#     ax.set_xlabel("Slip Ratio")
#     ax.set_ylabel(r"Normalized Friction Force ($\mu$)")
#     ax.grid(True)
#     ax.legend().remove()

# plt.show()
